# Initialization


In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

%matplotlib inline

#REPO = Path.cwd()

#if not (REPO / "new-results").exists():
 #   REPO = Path("/Users/ramontorres/Desktop/sustainability")
REPO = Path("/Users/ramontorres/Desktop/sustainability/new-results/")
RESULTS_DIR = REPO
ANALYSIS_DIR = REPO
FIGURES_DIR = ANALYSIS_DIR / "figures" / "section5"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO: {REPO}')
print(f'RESULTS_DIR: {RESULTS_DIR}')
print(f'RESULTS_DIR exists: {RESULTS_DIR.exists()}')
print(f'Contents: {list(RESULTS_DIR.iterdir())[:5]}')


NETWORKS = ["n1", "n2", "n4", "n5", "n6"]
NETWORK_RUN_IDS = {"n1": 1, "n2": 2, "n4": 4, "n5": 5, "n6": 6}

CONFIG_ORDER = [
    "baseline",
    "a0.20-k0.10-q1",
    "a0.30-k0.10-q1",
    "a0.50-k0.10-q1",
    "a1.0-k0.04-q1",
    "a1.0-k0.10-q1",
]
CONFIG_LABELS = {
    "baseline": "Baseline",
    "a0.20-k0.10-q1": "T90",
    "a0.30-k0.10-q1": "T85",
    "a0.50-k0.10-q1": "T80",
    "a1.0-k0.04-q1": "T75",
    "a1.0-k0.10-q1": "T70",
}
config_rank = {config: i for i, config in enumerate(CONFIG_ORDER)}

CONFIG_COLORS = {
    "baseline": "#3B3B3B",
    "a0.20-k0.10-q1": "#1F77B4",
    "a0.30-k0.10-q1": "#2CA02C",
    "a0.50-k0.10-q1": "#D62728",
    "a1.0-k0.04-q1": "#9467BD",
    "a1.0-k0.10-q1": "#E97451",
}
CONFIG_MARKERS = {
    "baseline": "o",
    "a0.20-k0.10-q1": "s",
    "a0.30-k0.10-q1": "^",
    "a0.50-k0.10-q1": "D",
    "a1.0-k0.04-q1": "v",
    "a1.0-k0.10-q1": "*",
}

FONT_SIZE = 22
LABEL_SIZE = 24
LEGEND_SIZE = 22
LINE_WIDTH = 3.2
MARKER_SIZE = 9

def save_figure(fig, out_path, **kwargs):
    out_path = Path(out_path)
    pdf_path = out_path.with_suffix(".pdf")
    fig.savefig(out_path, **kwargs)
    fig.savefig(pdf_path, **kwargs)
    return out_path, pdf_path


In [ ]:
from math import sqrt
SPINE_COLOR = 'gray'

def latexify(fig_width=None, fig_height=None, columns=1):
    """Set up matplotlib's RC params for LaTeX plotting.
    Call this before plotting a figure.
    Parameters
    ----------
    fig_width : float, optional, inches
    fig_height : float,  optional, inches
    columns : {1, 2}
    """

    # code adapted from http://www.scipy.org/Cookbook/Matplotlib/LaTeX_Examples

    # Width and max height in inches for IEEE journals taken from
    # computer.org/cms/Computer.org/Journal%20templates/transactions_art_guide.pdf

    assert(columns in [1,2])

    if fig_width is None:
        fig_width = 3.39 if columns==1 else 6.9 # width in inches

    if fig_height is None:
        golden_mean = (sqrt(5)-1.0)/2.0    # Aesthetic ratio
        fig_height = fig_width*golden_mean # height in inches

    MAX_HEIGHT_INCHES = 8.0
    if fig_height > MAX_HEIGHT_INCHES:
        print("WARNING: fig_height too large:" + fig_height +
              "so will reduce to" + MAX_HEIGHT_INCHES + "inches.")
        fig_height = MAX_HEIGHT_INCHES

    params = {'backend': 'ps',
              'text.latex.preamble': ['\\usepackage{gensymb}'],
              'axes.labelsize': 8, # fontsize for x and y labels (was 10)
              'axes.titlesize': 8,
              'font.size': 8, # was 10
              'legend.fontsize': 8, # was 10
              'xtick.labelsize': 8,
              'ytick.labelsize': 8,
              'text.usetex': True,
              'figure.figsize': [fig_width,fig_height],
              'font.family': 'times new roman'
    }

    #matplotlib.rcParams.update(params)
    return params


def format_axes(ax):

    for spine in ['top', 'right']:
        ax.spines[spine].set_color(SPINE_COLOR)
        ax.spines[spine].set_linewidth(0.5)

    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color(SPINE_COLOR)
        ax.spines[spine].set_linewidth(0.5)

    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')

    for axis in [ax.xaxis, ax.yaxis]:
        axis.set_tick_params(direction='in', color=SPINE_COLOR)

    return ax

latexify()

def getCDF(data):
    xdata = np.sort(data)
    ydata = [i/len(xdata) for i in range(len(xdata))]
    return xdata, ydata

def getCDFish(data):
    xdata = np.sort(data)
    ydata = [i for i in range(len(xdata))]
    return xdata, ydata

import re
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    '''
    alist.sort(key=natural_keys) sorts in human order
    http://nedbatchelder.com/blog/200712/human_sorting.html
    (See Toothy's implementation in the comments)
    '''
    return [ atoi(c) for c in re.split(r'(\d+)', text) ]

TIMESTAMP_STR_FORMATS = [
    '%Y%m%d_%H:%M:%S',
    '%Y-%m-%d %H:%M:%S.%f',
    '%Y%m%d %H:%M:%S',
    '%Y-%m-%d_%H:%M:%S',
    '%Y-%m-%dT%H:%M:%S.%f',
    '%H:%M:%S.%f',
    '%Y-%m-%dT%H:%M:%S.%f%z',
    '%Y%m%dT%H:%M:%S',
    '%Y-%m-%d_%H:%M:%S.%f',
    '%Y%m%d_%H:%M:%S.%f',
    '%Y-%m-%dT%H:%M:%S',
    '%Y-%m-%d %H:%M:%S',
    '%Y%m%dT%H:%M:%S.%f',
    '%H:%M:%S',
    '%Y%m%d %H:%M:%S.%f']

def to_epoch(t_str):
    """
    Covert a timestamp string to an epoch number.
    :param str t_str: a timestamp string.
    :return int: epoch number of the timestamp.
    """
    try:
        t = float(t_str)
        return t
    except ValueError:
        for format in TIMESTAMP_STR_FORMATS:
            try:
                t = datetime.datetime.strptime(t_str, format)
                return float(time.mktime(t.utctimetuple()) * 1000.0 + t.microsecond / 1000.0)
            except ValueError:
                pass
    raise exceptions.InvalidDataFormat


## Load data

In [ ]:
# Config paths are a little different for n4, so normalize them here.
config_paths = {
    "n4": {
        "baseline": RESULTS_DIR / "n4" / "baseline",
        "a0.20-k0.10-q1": RESULTS_DIR / "n4" / "a0.20-k0.10-q1",
        "a0.30-k0.10-q1": RESULTS_DIR / "n4" / "a0.30-k0.10-q1",
        "a0.50-k0.10-q1": RESULTS_DIR / "n4"  / "a0.50-k0.10-q1",
        "a1.0-k0.04-q1": RESULTS_DIR / "n4"  / "a1.0-k0.04-q1",
        "a1.0-k0.10-q1": RESULTS_DIR / "n4" / "a1.0-k0.10-q1",
    }
}

for network in ["n1", "n2", "n5", "n6"]:
    base = RESULTS_DIR / f"{network}"
    config_paths[network] = {config: base / config for config in CONFIG_ORDER}


def hour_dirs_for(network, config_dir):
    run_id = NETWORK_RUN_IDS[network]
    hour_dirs = [
        p for p in config_dir.glob("*_Hour_*")
        if re.search(rf"_Run_{run_id}_Hour_\d+$", p.name)
    ]
    if not hour_dirs:
        hour_dirs = list(config_dir.glob("*_Hour_*"))
    return sorted(
        hour_dirs,
        key=lambda p: int(re.search(r"_Hour_(\d+)$", p.name).group(1)),
    )


In [ ]:

# One row per network/config/hour.
summary_rows = []
missing_rows = []

for network in NETWORKS:
    for config in CONFIG_ORDER:
        config_dir = config_paths[network][config]
        hour_dirs = hour_dirs_for(network, config_dir)

        for hour_dir in hour_dirs:
            match = re.search(r"_Hour_(\d+)$", hour_dir.name)
            if not match:
                continue
            hour = int(match.group(1))
            emissions_path = hour_dir / "cache_emissions.csv"
            tgen_path = hour_dir / "cache_tgen.json"

            if not emissions_path.exists():
                missing_rows.append({"network": network, "config": config, "hour": hour, "missing": "cache_emissions.csv"})
                continue

            emissions = pd.read_csv(emissions_path)
            tgen = {}
            if tgen_path.exists():
                tgen = json.loads(tgen_path.read_text())

            summary_rows.append({
                "network": network,
                "config": config,
                "config_label": CONFIG_LABELS[config],
                "hour": hour,
                "relay_rows": len(emissions),
                "throughput_bytes": emissions["throughput"].sum(),
                "carbon_emissions": emissions["carbon_emissions"].sum(),
                "energy_kwh": emissions["energy_kwh"].sum(),
                "circuits_built": emissions["circuits_built"].sum() if "circuits_built" in emissions else np.nan,
                "tgen_perf_actual": tgen.get("perf_actual", np.nan),
                "tgen_markov_recv": tgen.get("markov_recv", np.nan),
                "source_dir": str(hour_dir.relative_to(REPO)),
            })

master_df = pd.DataFrame(summary_rows)
master_df["throughput_gib"] = master_df["throughput_bytes"] / 1024**3
master_df["network_label"] = master_df["network"].str.upper()

missing_df = pd.DataFrame(missing_rows)

print(f"Loaded {len(master_df):,} hourly config rows")
print(f"Missing rows: {len(missing_df):,}")
master_df.head()


In [ ]:
# Compare each config to the same network's baseline in the same hour.
baseline_hourly = (
    master_df[master_df["config"] == "baseline"]
    [["network", "hour", "throughput_bytes", "carbon_emissions"]]
    .rename(columns={
        "throughput_bytes": "baseline_throughput_bytes",
        "carbon_emissions": "baseline_carbon_emissions",
    })
)

relative_df = master_df.merge(baseline_hourly, on=["network", "hour"], how="left")
relative_df["throughput_pct_baseline"] = relative_df["throughput_bytes"] / relative_df["baseline_throughput_bytes"] * 100
relative_df["carbon_pct_baseline"] = relative_df["carbon_emissions"] / relative_df["baseline_carbon_emissions"] * 100

relative_df.head()


# Treat networks the same: average the per-network baseline-relative values by hour/config.
time_series_df = (
    relative_df.groupby(["config", "config_label", "hour"], as_index=False)
    .agg(
        networks=("network", "nunique"),
        throughput_pct_baseline=("throughput_pct_baseline", "mean"),
        throughput_pct_std=("throughput_pct_baseline", "std"),
        carbon_pct_baseline=("carbon_pct_baseline", "mean"),
        carbon_pct_std=("carbon_pct_baseline", "std"),
    )
)

time_series_df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)

for config in CONFIG_ORDER:
    cfg = time_series_df[time_series_df["config"] == config].sort_values("hour")
    label = CONFIG_LABELS[config]
    color = CONFIG_COLORS[config]
    marker = CONFIG_MARKERS[config]

    hours = cfg["hour"].to_numpy()
    throughput_mean = cfg["throughput_pct_baseline"].to_numpy()
    throughput_std = cfg["throughput_pct_std"].fillna(0).to_numpy()
    carbon_mean = cfg["carbon_pct_baseline"].to_numpy()
    carbon_std = cfg["carbon_pct_std"].fillna(0).to_numpy()

    axes[0].fill_between(
        hours, throughput_mean - throughput_std, throughput_mean + throughput_std,
        color=color, alpha=0.14, linewidth=0,
    )
    axes[0].plot(
        hours, throughput_mean,
        label=label, color=color, marker=marker, linewidth=LINE_WIDTH, markersize=MARKER_SIZE,
    )
    axes[1].fill_between(
        hours, carbon_mean - carbon_std, carbon_mean + carbon_std,
        color=color, alpha=0.14, linewidth=0,
    )
    axes[1].plot(
        hours, carbon_mean,
        label=label, color=color, marker=marker, linewidth=LINE_WIDTH, markersize=MARKER_SIZE,
    )

for ax, ylabel in zip(axes, ["Data\nTransferred", "Carbon\nFootprint"]):
    ax.axhline(100, color="#666666", linestyle="--", linewidth=1.8, alpha=0.7)
    ax.set_ylabel(ylabel, fontsize=LABEL_SIZE, rotation=90)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.tick_params(axis="both", labelsize=FONT_SIZE)
    format_axes(ax)

axes[0].set_ylim(bottom=75)
axes[0].set_yticks([80, 90, 100])
axes[1].set_ylim(bottom=0, top=105)
axes[1].set_yticks([0, 25, 50, 75, 100])
axes[1].set_xlabel("Hour of day", fontsize=LABEL_SIZE, labelpad=12)
axes[1].set_xticks(range(0, 24, 3))
axes[1].set_xlim(-0.3, 23.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=5, loc="upper center", bbox_to_anchor=(0.5, 0.99), frameon=True, fontsize=LEGEND_SIZE, handlelength=1.8, columnspacing=1.0)

fig.subplots_adjust(left=0.18, right=0.99, bottom=0.15, top=0.80, hspace=0.36)
out_path = FIGURES_DIR / "data_carbon_pct_baseline_over_time.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()


## Completion and congestion CDFs

CDFs for stream completion time and relay write congestion. The congestion plot keeps one main CDF and uses an inset to zoom into the high-percentile tail.

In [ ]:
# Load completion times and daily per-relay congestion.
cdf_labels = {**CONFIG_LABELS, "baseline": "Base."}
completion_parts = []
congestion_hour_parts = []
congestion_skipped_rows = []

for network in NETWORKS:
    for config in CONFIG_ORDER:
        config_dir = config_paths[network][config]
        hour_dirs = hour_dirs_for(network, config_dir)

        for hour_dir in hour_dirs:
            match = re.search(r"_Hour_(\d+)$", hour_dir.name)
            if not match:
                continue
            hour = int(match.group(1))

            completion_path = hour_dir / "cache_completion_times.json"
            if completion_path.exists():
                times = json.loads(completion_path.read_text())
                if times:
                    completion_parts.append(pd.DataFrame({
                        "network": network,
                        "config": config,
                        "config_label": CONFIG_LABELS[config],
                        "hour": hour,
                        "time_s": times,
                    }))

            emissions_path = hour_dir / "cache_emissions.csv"
            if not emissions_path.exists():
                continue
            columns = pd.read_csv(emissions_path, nrows=0).columns
            needed = {"relay", "write_congested_s", "read_congested_s"}
            if not needed.issubset(columns):
                congestion_skipped_rows.append({
                    "network": network,
                    "config": config,
                    "hour": hour,
                    "source_dir": str(hour_dir.relative_to(REPO)),
                })
                continue

            congestion_hour = pd.read_csv(
                emissions_path,
                usecols=["relay", "write_congested_s", "read_congested_s"],
            )
            congestion_hour["network"] = network
            congestion_hour["config"] = config
            congestion_hour["config_label"] = CONFIG_LABELS[config]
            congestion_hour["hour"] = hour
            congestion_hour_parts.append(congestion_hour)

completion_df = pd.concat(completion_parts, ignore_index=True)
completion_df["config_label"] = completion_df["config"].map(CONFIG_LABELS)

congestion_hour_df = pd.concat(congestion_hour_parts, ignore_index=True)
congestion_relay_df = (
    congestion_hour_df.groupby(["network", "config", "config_label", "relay"], as_index=False)
    .agg(
        write_congested_s=("write_congested_s", "sum"),
        read_congested_s=("read_congested_s", "sum"),
    )
)
congestion_relay_df["total_congested_s"] = congestion_relay_df["write_congested_s"] + congestion_relay_df["read_congested_s"]
congestion_skipped_df = pd.DataFrame(congestion_skipped_rows)

print(f"Completion observations: {len(completion_df):,}")
print(f"Daily relay congestion rows: {len(congestion_relay_df):,}")
print(f"Skipped congestion files: {len(congestion_skipped_df):,}")
completion_df.head()


In [ ]:
# Figure 11: completion time CDF.
fig, ax = plt.subplots(figsize=(8.8, 5.3))

for config in CONFIG_ORDER:
    times = np.sort(completion_df.loc[completion_df["config"] == config, "time_s"].to_numpy())
    if len(times) == 0:
        continue
    y = np.arange(1, len(times) + 1) / len(times)
    mark_every = max(1, len(times) // 35)
    ax.plot(
        times, y,
        color=CONFIG_COLORS[config], linewidth=LINE_WIDTH,
        marker=CONFIG_MARKERS[config], markersize=7, markevery=mark_every,
        label=f"{cdf_labels[config]} (n={len(times):,})",
    )

ax.set_xlim(0, 60)
ax.set_ylim(0, 1.01)
ax.set_xlabel("Completion time (s)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Fraction of streams", fontsize=LABEL_SIZE, labelpad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.tick_params(axis="both", labelsize=FONT_SIZE)
format_axes(ax)

ax.legend(loc="lower right", frameon=True, fontsize=18)
fig.tight_layout()

out_path = FIGURES_DIR / "completion_time_cdf.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()


In [ ]:
# Figure 12: write congestion CDF with tail inset.
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

fig, ax = plt.subplots(figsize=(8.5, 5))
axins = inset_axes(ax, width="62%", height="62%", loc="lower right", borderpad=1.4)

all_tail_vals = []
for config in CONFIG_ORDER:
    vals = np.sort(congestion_relay_df.loc[congestion_relay_df["config"] == config, "write_congested_s"].to_numpy())
    if len(vals) == 0:
        continue
    y = np.arange(1, len(vals) + 1) / len(vals)
    all_tail_vals.extend(vals)
    mark_every = max(1, len(vals) // 35)

    ax.plot(vals, y, color=CONFIG_COLORS[config], linewidth=LINE_WIDTH, marker=CONFIG_MARKERS[config], markersize=7, markevery=mark_every, label=cdf_labels[config])
    axins.plot(vals, y, color=CONFIG_COLORS[config], linewidth=2.2, marker=CONFIG_MARKERS[config], markersize=4, markevery=mark_every)

tail_xmax = float(np.ceil(np.nanpercentile(all_tail_vals, 99.95) / 1000) * 1000)
tail_xmax = max(1000, tail_xmax)

ax.set_xlim(0, tail_xmax)
ax.set_ylim(0, 1.01)
ax.set_xlabel("Write-congested seconds per relay (24 h)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Fraction of relays", fontsize=LABEL_SIZE, labelpad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.tick_params(axis="both", labelsize=FONT_SIZE)
format_axes(ax)

axins.set_xlim(0, tail_xmax)
axins.set_ylim(0.9, 1.002)
axins.set_title("Tail zoom", fontsize=16, pad=4)
axins.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axins.set_yticks([0.9, 0.95, 1.0])
axins.grid(axis="both", linestyle="--", alpha=0.25)
axins.tick_params(axis="both", labelsize=FONT_SIZE-4)
format_axes(axins)
#mark_inset(ax, axins, loc1=2, loc2=4, fc="none", ec="0.35", linewidth=1.0)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="upper center", bbox_to_anchor=(0.5, 0.99), frameon=True, fontsize=18)
fig.subplots_adjust(left=0.16, right=0.98, bottom=0.18, top=0.82)

out_path = FIGURES_DIR / "relay_congestion_cdf_tail_inset.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()


## Exploring

In [ ]:
# Average each network's hourly percentages first, then average across networks.
tradeoff_network_df = (
    relative_df.groupby(["network", "config", "config_label"], as_index=False)
    .agg(
        data_pct_baseline=("throughput_pct_baseline", "mean"),
        carbon_pct_baseline=("carbon_pct_baseline", "mean"),
        hours=("hour", "nunique"),
    )
)

tradeoff_df = (
    tradeoff_network_df.groupby(["config", "config_label"], as_index=False)
    .agg(
        networks=("network", "nunique"),
        hours_min=("hours", "min"),
        data_pct_baseline=("data_pct_baseline", "mean"),
        data_pct_std=("data_pct_baseline", "std"),
        carbon_pct_baseline=("carbon_pct_baseline", "mean"),
        carbon_pct_std=("carbon_pct_baseline", "std"),
    )
)
tradeoff_df["data_loss_pct"] = 100 - tradeoff_df["data_pct_baseline"]
tradeoff_df["carbon_saved_pct"] = 100 - tradeoff_df["carbon_pct_baseline"]
tradeoff_df["carbon_saved_per_data_loss"] = np.where(
    tradeoff_df["data_loss_pct"] > 0,
    tradeoff_df["carbon_saved_pct"] / tradeoff_df["data_loss_pct"],
    np.nan,
)
tradeoff_df["carbon_per_gib_pct_baseline"] = (
    tradeoff_df["carbon_pct_baseline"] / tradeoff_df["data_pct_baseline"] * 100
)

tradeoff_df["config_rank"] = tradeoff_df["config"].map({config: i for i, config in enumerate(CONFIG_ORDER)})
tradeoff_df = tradeoff_df.sort_values("config_rank").drop(columns="config_rank")

In [ ]:
# Plot: carbon savings versus data loss. Points above the diagonal are good tradeoffs.
fig, ax = plt.subplots(figsize=(7.5, 4.5))

xmax = max(15, float(np.ceil(tradeoff_df["data_loss_pct"].max() / 5) * 5))
ymax = 100
ax.plot([0, xmax], [0, xmax], color="#777777", linestyle="--", linewidth=1.8, alpha=0.8, label="1:1")

for _, row in tradeoff_df.iterrows():
    config = row["config"]
    label = "Base." if config == "baseline" else CONFIG_LABELS[config]
    x = row["data_loss_pct"]
    y = row["carbon_saved_pct"]
    xerr = 0 if pd.isna(row["data_pct_std"]) else row["data_pct_std"]
    yerr = 0 if pd.isna(row["carbon_pct_std"]) else row["carbon_pct_std"]

    plotline, caps, bars = ax.errorbar(
        x, y, xerr=xerr, yerr=yerr,
        fmt=CONFIG_MARKERS[config], markersize=5,
        color=CONFIG_COLORS[config], ecolor=CONFIG_COLORS[config],
        elinewidth=1.0, capsize=5, label=label,
    )
    ax.text(x + 0.35, y + 1.8, label, fontsize=20, color=CONFIG_COLORS[config])
    plotline.set_alpha(0.4) 


# Set transparency only for the error lines and caps

ax.set_xlim(-0.5, xmax)
ax.set_ylim(-2, ymax)
ax.set_xlabel("Loss in Data Transfer (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Carbon Saved (%)", fontsize=LABEL_SIZE, labelpad=12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.tick_params(axis="both", labelsize=FONT_SIZE)
format_axes(ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:], labels[1:], loc="lower right", frameon=True, fontsize=20)
fig.subplots_adjust(left=0.20, right=0.98, bottom=0.18, top=0.98)

out_path = FIGURES_DIR / "headline_carbon_saved_vs_data_loss.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()


## Country breakdown

In [ ]:
# One row per country/network/config/hour.
country_hour_parts = []

for network in NETWORKS:
    for config in CONFIG_ORDER:
        config_dir = config_paths[network][config]
        hour_dirs = hour_dirs_for(network, config_dir)

        for hour_dir in hour_dirs:
            match = re.search(r"_Hour_(\d+)$", hour_dir.name)
            if not match:
                continue
            emissions_path = hour_dir / "cache_emissions.csv"
            if not emissions_path.exists():
                continue

            emissions = pd.read_csv(
                emissions_path,
                usecols=["country", "throughput", "carbon_emissions", "carbon_intensity"],
            )
            emissions["country"] = emissions["country"].fillna("Unknown").astype(str)

            country_hour = (
                emissions.groupby("country", as_index=False)
                .agg(
                    throughput_bytes=("throughput", "sum"),
                    carbon_emissions=("carbon_emissions", "sum"),
                    carbon_intensity=("carbon_intensity", "mean"),
                )
            )
            country_hour["network"] = network
            country_hour["config"] = config
            country_hour["config_label"] = CONFIG_LABELS[config]
            country_hour["hour"] = int(match.group(1))
            country_hour_parts.append(country_hour)

country_hour_df = pd.concat(country_hour_parts, ignore_index=True)
print(f"Country-hour rows: {len(country_hour_df):,}")
country_hour_df.head()


In [ ]:
# Daily country shares, with missing countries counted as zero for each network/config.
country_daily_raw = (
    country_hour_df.groupby(["network", "config", "config_label", "country"], as_index=False)
    .agg(
        throughput_bytes=("throughput_bytes", "sum"),
        carbon_emissions=("carbon_emissions", "sum"),
        carbon_intensity=("carbon_intensity", "mean"),
    )
)

all_countries = sorted(country_hour_df["country"].dropna().unique())
full_country_grid = pd.MultiIndex.from_product(
    [NETWORKS, CONFIG_ORDER, all_countries],
    names=["network", "config", "country"],
).to_frame(index=False)

country_ci = country_hour_df.groupby("country")["carbon_intensity"].mean()
country_daily_df = full_country_grid.merge(
    country_daily_raw,
    on=["network", "config", "country"],
    how="left",
)
country_daily_df["config_label"] = country_daily_df["config"].map(CONFIG_LABELS)
country_daily_df["throughput_bytes"] = country_daily_df["throughput_bytes"].fillna(0)
country_daily_df["carbon_emissions"] = country_daily_df["carbon_emissions"].fillna(0)
country_daily_df["carbon_intensity"] = country_daily_df["country"].map(country_ci)

country_daily_df["data_share"] = (
    country_daily_df["throughput_bytes"]
    / country_daily_df.groupby(["network", "config"])["throughput_bytes"].transform("sum")
    * 100
)
country_daily_df["carbon_share"] = (
    country_daily_df["carbon_emissions"]
    / country_daily_df.groupby(["network", "config"])["carbon_emissions"].transform("sum")
    * 100
)

country_avg_df = (
    country_daily_df.groupby(["config", "config_label", "country"], as_index=False)
    .agg(
        data_share=("data_share", "mean"),
        carbon_share=("carbon_share", "mean"),
        carbon_intensity=("carbon_intensity", "mean"),
    )
)
country_avg_df["carbon_minus_data_pp"] = country_avg_df["carbon_share"] - country_avg_df["data_share"]

country_config_totals_df = (
    country_daily_df.groupby(["network", "config", "config_label"], as_index=False)
    .agg(
        throughput_bytes=("throughput_bytes", "sum"),
        carbon_emissions=("carbon_emissions", "sum"),
    )
)
country_config_baseline_df = (
    country_config_totals_df[country_config_totals_df["config"] == "baseline"]
    [["network", "throughput_bytes", "carbon_emissions"]]
    .rename(columns={
        "throughput_bytes": "baseline_throughput_bytes",
        "carbon_emissions": "baseline_carbon_emissions",
    })
)
country_config_totals_df = country_config_totals_df.merge(country_config_baseline_df, on="network", how="left")
country_config_totals_df["throughput_pct_baseline"] = country_config_totals_df["throughput_bytes"] / country_config_totals_df["baseline_throughput_bytes"] * 100
country_config_totals_df["carbon_pct_baseline"] = country_config_totals_df["carbon_emissions"] / country_config_totals_df["baseline_carbon_emissions"] * 100

# Match the main time-series figure: average hourly network-normalized percentages.
country_total_summary_df = (
    time_series_df.groupby(["config", "config_label"], as_index=False)
    .agg(
        throughput_pct_baseline=("throughput_pct_baseline", "mean"),
        carbon_pct_baseline=("carbon_pct_baseline", "mean"),
    )
)

country_rank_df = (
    country_avg_df.groupby("country", as_index=False)
    .agg(
        max_carbon_share=("carbon_share", "max"),
        avg_carbon_share=("carbon_share", "mean"),
        carbon_intensity=("carbon_intensity", "mean"),
    )
)
baseline_carbon_share = (
    country_avg_df[country_avg_df["config"] == "baseline"]
    [["country", "carbon_share"]]
    .rename(columns={"carbon_share": "baseline_carbon_share"})
)
country_rank_df = country_rank_df.merge(baseline_carbon_share, on="country", how="left")
country_rank_df["baseline_carbon_share"] = country_rank_df["baseline_carbon_share"].fillna(0)

TOP_COUNTRIES = (
    country_rank_df.sort_values(["max_carbon_share", "baseline_carbon_share"], ascending=False)
    .head(9)["country"]
    .tolist()
)

palette = list(plt.get_cmap("tab20").colors)
COUNTRY_COLORS = {country: palette[i % len(palette)] for i, country in enumerate(TOP_COUNTRIES)}
COUNTRY_COLORS["Other"] = "#c9c9c9"
COUNTRY_HATCHES = {
    country: hatch
    for country, hatch in zip(TOP_COUNTRIES, ["/", "x", ".", "o", "+", "-", "**", "OO", "||"])
}
COUNTRY_HATCHES["Other"] = ""
COUNTRY_LABELS = {
    country: f"{country} ({country_ci.get(country, np.nan):.0f})"
    for country in TOP_COUNTRIES
}
COUNTRY_LABELS["Other"] = "Other"

print("Top countries shown:", TOP_COUNTRIES)
country_avg_df[country_avg_df["country"].isin(TOP_COUNTRIES)].head()


In [ ]:
# Paired stacked bar: data composition vs carbon composition.
stacked_bar_labels = {**CONFIG_LABELS, "baseline": "Base."}
x = np.arange(len(CONFIG_ORDER))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)

for ax, metric, title, total_col in [
    (axes[0], "data_share", "Data Transferred", "throughput_pct_baseline"),
    (axes[1], "carbon_share", "Carbon Footprint", "carbon_pct_baseline"),
]:
    bottoms = np.zeros(len(CONFIG_ORDER))

    for country in TOP_COUNTRIES:
        vals = (
            country_avg_df[country_avg_df["country"] == country]
            .set_index("config")
            .reindex(CONFIG_ORDER)[metric]
            .fillna(0)
            .to_numpy()
        )
        ax.bar(
            x, vals, bottom=bottoms, width=0.72,
            color=COUNTRY_COLORS[country], edgecolor="white", linewidth=0.5,
            hatch=COUNTRY_HATCHES[country],
            label=COUNTRY_LABELS[country],
        )
        bottoms += vals

    other_vals = np.maximum(0, 100 - bottoms)
    ax.bar(
        x, other_vals, bottom=bottoms, width=0.72,
        color=COUNTRY_COLORS["Other"], edgecolor="white", linewidth=0.5,
        label=COUNTRY_LABELS["Other"],
    )

    total_labels = (
        country_total_summary_df.set_index("config")
        .reindex(CONFIG_ORDER)[total_col]
        .round(0)
        .astype(int)
    )
    for i, pct in enumerate(total_labels):
        ax.text(i, 103, f"{pct}%", ha="center", va="bottom", fontsize=16)

    ax.set_title(title, fontsize=LABEL_SIZE, pad=14)
    ax.set_xticks(x)
    ax.set_xticklabels([stacked_bar_labels[c] for c in CONFIG_ORDER], rotation=0, fontsize=18)
    ax.set_ylim(0, 112)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.tick_params(axis="y", labelsize=18)
    format_axes(ax)

axes[0].set_ylabel("Share of total", fontsize=LABEL_SIZE, labelpad=12)
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(
    handles, labels, ncol=5, loc="upper center", bbox_to_anchor=(0.45, 1.35),
    frameon=True, fontsize=14, title="Country", title_fontsize=16,
)
#fig.text(0.5, -0.02, "Numbers above bars show total relative to baseline", ha="center", fontsize=16)
#fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.subplots_adjust(right=None, bottom=None, left=None, top=None, wspace=0.3, hspace=None)


out_path = FIGURES_DIR / "country_data_carbon_stacked_bar.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
traffic_share_de_us = (
    country_avg_df[country_avg_df["country"].isin(["DE", "US"])]
    .pivot(index="config", columns="country", values="data_share")
    .reindex(CONFIG_ORDER)
)
print(traffic_share_de_us.round(2))
# Combined carbon emissions share (US + Germany), by config.
carbon_share_de_us = (
    country_avg_df[country_avg_df["country"].isin(["DE", "US"])]
    .groupby("config")["carbon_share"]
    .sum()
    .reindex(CONFIG_ORDER)
)
print(carbon_share_de_us.round(2))
plt.show()


## Reliability, concentration, and dirty-grid traffic

In [ ]:
# Load relay-hour rows and circuit failure counts.
relay_hour_parts = []
failure_hour_rows = []
relay_columns = [
    "relay", "country", "circuit_count", "circuits_built", "throughput",
    "bandwidth_per_circuit", "old_bw_weight", "new_bw_weight",
    "carbon_intensity", "error_count", "bw_rate",
    "write_congested_s", "read_congested_s", "carbon_emissions",
]

for network in NETWORKS:
    for config in CONFIG_ORDER:
        config_dir = config_paths[network][config]
        hour_dirs = hour_dirs_for(network, config_dir)

        for hour_dir in hour_dirs:
            match = re.search(r"_Hour_(\d+)$", hour_dir.name)
            if not match:
                continue
            hour = int(match.group(1))

            emissions_path = hour_dir / "cache_emissions.csv"
            if emissions_path.exists():
                available_cols = pd.read_csv(emissions_path, nrows=0).columns
                usecols = [col for col in relay_columns if col in available_cols]
                relay_hour = pd.read_csv(emissions_path, usecols=usecols)
                for col in relay_columns:
                    if col not in relay_hour.columns:
                        relay_hour[col] = np.nan
                relay_hour["country"] = relay_hour["country"].fillna("Unknown").astype(str)
                relay_hour["network"] = network
                relay_hour["config"] = config
                relay_hour["config_label"] = CONFIG_LABELS[config]
                relay_hour["hour"] = hour
                relay_hour_parts.append(relay_hour)

            failures_path = hour_dir / "cache_circuit_failures.json"
            if failures_path.exists():
                failures = json.loads(failures_path.read_text())
                row = {
                    "network": network,
                    "config": config,
                    "config_label": CONFIG_LABELS[config],
                    "hour": hour,
                    "failure_count": sum(failures.values()),
                }
                for reason, count in failures.items():
                    row[f"failure_{reason.lower()}"] = count
                failure_hour_rows.append(row)

relay_hour_df = pd.concat(relay_hour_parts, ignore_index=True)
for col in [
    "circuit_count", "circuits_built", "throughput", "bandwidth_per_circuit",
    "old_bw_weight", "new_bw_weight", "carbon_intensity", "error_count",
    "bw_rate", "write_congested_s", "read_congested_s", "carbon_emissions",
]:
    relay_hour_df[col] = pd.to_numeric(relay_hour_df[col], errors="coerce").fillna(0)

failure_hour_df = pd.DataFrame(failure_hour_rows).fillna(0)
print(f"Relay-hour rows: {len(relay_hour_df):,}")
print(f"Failure-hour rows: {len(failure_hour_df):,}")
relay_hour_df.head()


This analysis below can likely be included in text

In [ ]:
# Load concentration summary. Effective count is 1 / sum(share^2), so higher means less concentration.
relay_daily_df = (
    relay_hour_df.groupby(["network", "config", "config_label", "relay"], as_index=False)
    .agg(throughput_bytes=("throughput", "sum"))
)
relay_daily_df["share"] = (
    relay_daily_df["throughput_bytes"]
    / relay_daily_df.groupby(["network", "config"])["throughput_bytes"].transform("sum")
)

relay_concentration_network_df = (
    relay_daily_df.groupby(["network", "config", "config_label"], as_index=False)
    .agg(
        relays=("relay", "nunique"),
        relay_hhi=("share", lambda s: (s ** 2).sum() * 10000),
        effective_relays=("share", lambda s: 1 / (s ** 2).sum()),
        top10_relay_share=("share", lambda s: s.nlargest(10).sum() * 100),
    )
)

country_daily_load_df = (
    relay_hour_df.groupby(["network", "config", "config_label", "country"], as_index=False)
    .agg(throughput_bytes=("throughput", "sum"))
)
country_daily_load_df["share"] = (
    country_daily_load_df["throughput_bytes"]
    / country_daily_load_df.groupby(["network", "config"])["throughput_bytes"].transform("sum")
)

country_concentration_network_df = (
    country_daily_load_df.groupby(["network", "config", "config_label"], as_index=False)
    .agg(
        countries=("country", "nunique"),
        country_hhi=("share", lambda s: (s ** 2).sum() * 10000),
        effective_countries=("share", lambda s: 1 / (s ** 2).sum()),
        top3_country_share=("share", lambda s: s.nlargest(3).sum() * 100),
    )
)

concentration_network_df = relay_concentration_network_df.merge(
    country_concentration_network_df.drop(columns="config_label"),
    on=["network", "config"],
    how="left",
).merge(
    tradeoff_network_df[["network", "config", "carbon_pct_baseline"]],
    on=["network", "config"],
    how="left",
)
concentration_network_df["carbon_saved_pct"] = 100 - concentration_network_df["carbon_pct_baseline"]

concentration_summary_df = (
    concentration_network_df.groupby(["config", "config_label"], as_index=False)
    .agg(
        carbon_saved_pct=("carbon_saved_pct", "mean"),
        carbon_saved_std=("carbon_saved_pct", "std"),
        effective_relays=("effective_relays", "mean"),
        effective_relays_std=("effective_relays", "std"),
        top10_relay_share=("top10_relay_share", "mean"),
        effective_countries=("effective_countries", "mean"),
        effective_countries_std=("effective_countries", "std"),
        top3_country_share=("top3_country_share", "mean"),
    )
)
concentration_summary_df["label"] = concentration_summary_df["config"].map({**CONFIG_LABELS, "baseline": "Base."})
concentration_summary_df["config_rank"] = concentration_summary_df["config"].map({config: i for i, config in enumerate(CONFIG_ORDER)})
concentration_summary_df = concentration_summary_df.sort_values("config_rank").drop(columns="config_rank")

concentration_display_df = pd.DataFrame({
    "Setting": concentration_summary_df["label"],
    "Carbon saved": concentration_summary_df["carbon_saved_pct"].map(lambda v: f"{v:.1f}%"),
    "Effective relays": concentration_summary_df["effective_relays"].map(lambda v: f"{v:.0f}"),
    "Top-10 relay share": concentration_summary_df["top10_relay_share"].map(lambda v: f"{v:.1f}%"),
    "Effective countries": concentration_summary_df["effective_countries"].map(lambda v: f"{v:.1f}"),
    "Top-3 country share": concentration_summary_df["top3_country_share"].map(lambda v: f"{v:.1f}%"),
})

concentration_csv_path = FIGURES_DIR / "load_concentration_summary.csv"
concentration_display_df.to_csv(concentration_csv_path, index=False)
print(concentration_csv_path)
concentration_display_df


In [ ]:
# Share of traffic routed through countries above dirty-grid thresholds.
dirty_thresholds = [150, 300, 400, 600]
high_carbon_rows = []

for (network, config, config_label), sub in relay_hour_df.groupby(["network", "config", "config_label"]):
    row = {
        "network": network,
        "config": config,
        "config_label": config_label,
        "throughput_bytes": sub["throughput"].sum(),
    }
    for threshold in dirty_thresholds:
        row[f"traffic_ge_{threshold}_bytes"] = sub.loc[
            sub["carbon_intensity"] >= threshold, "throughput"
        ].sum()
    high_carbon_rows.append(row)

high_carbon_network_df = pd.DataFrame(high_carbon_rows)
for threshold in dirty_thresholds:
    high_carbon_network_df[f"traffic_ge_{threshold}_share"] = (
        high_carbon_network_df[f"traffic_ge_{threshold}_bytes"]
        / high_carbon_network_df["throughput_bytes"] * 100
    )

high_carbon_network_df = high_carbon_network_df.merge(
    tradeoff_network_df[["network", "config", "carbon_pct_baseline"]],
    on=["network", "config"],
    how="left",
)
high_carbon_network_df["carbon_saved_pct"] = 100 - high_carbon_network_df["carbon_pct_baseline"]

agg_spec = {
    "carbon_saved_pct": ("carbon_saved_pct", "mean"),
    "carbon_saved_std": ("carbon_saved_pct", "std"),
}
for threshold in dirty_thresholds:
    agg_spec[f"traffic_ge_{threshold}_share"] = (f"traffic_ge_{threshold}_share", "mean")
    agg_spec[f"traffic_ge_{threshold}_share_std"] = (f"traffic_ge_{threshold}_share", "std")

high_carbon_summary_df = (
    high_carbon_network_df.groupby(["config", "config_label"], as_index=False)
    .agg(**agg_spec)
)
high_carbon_summary_df["label"] = high_carbon_summary_df["config"].map({**CONFIG_LABELS, "baseline": "Base."})
high_carbon_summary_df["config_rank"] = high_carbon_summary_df["config"].map({config: i for i, config in enumerate(CONFIG_ORDER)})
high_carbon_summary_df = high_carbon_summary_df.sort_values("config_rank").drop(columns="config_rank")

high_carbon_display_df = pd.DataFrame({
    "Setting": high_carbon_summary_df["label"],
    "Carbon saved": high_carbon_summary_df["carbon_saved_pct"].map(lambda v: f"{v:.1f}%"),
    ">=150": high_carbon_summary_df["traffic_ge_150_share"].map(lambda v: f"{v:.1f}%"),
    ">=300": high_carbon_summary_df["traffic_ge_300_share"].map(lambda v: f"{v:.1f}%"),
    ">=400": high_carbon_summary_df["traffic_ge_400_share"].map(lambda v: f"{v:.1f}%"),
    ">=600": high_carbon_summary_df["traffic_ge_600_share"].map(lambda v: f"{v:.1f}%"),
})

high_carbon_csv_path = FIGURES_DIR / "dirty_grid_traffic_share_summary.csv"
high_carbon_display_df.to_csv(high_carbon_csv_path, index=False)
print(high_carbon_csv_path)
high_carbon_display_df


In [ ]:
# Plot: the optimizer quickly removes traffic from high-carbon grids.
fig, ax = plt.subplots(figsize=(8.8, 5.0))
threshold_colors = {150: "#1F77B4", 300: "#D62728", 400: "#FF7F0E", 600: "#8C564B"}
threshold_markers = {150: "D", 300: "o", 400: "s", 600: "^"}

for threshold in dirty_thresholds:
    y_col = f"traffic_ge_{threshold}_share"
    yerr_col = f"traffic_ge_{threshold}_share_std"
    plotline, caps, bars = ax.errorbar(
        high_carbon_summary_df["carbon_saved_pct"], high_carbon_summary_df[y_col],
        xerr=high_carbon_summary_df["carbon_saved_std"].fillna(0),
        yerr=high_carbon_summary_df[yerr_col].fillna(0),
        marker=threshold_markers[threshold], markersize=10,
        linewidth=2.8, capsize=4,
        color=threshold_colors[threshold],
        label=f">= {threshold} gCO2/kWh",
    )
    plotline.set_alpha(0.6) 

for _, row in high_carbon_summary_df.iterrows():
    ax.text(
        row["carbon_saved_pct"] + 1.0, row["traffic_ge_150_share"] + 0.35,
        row["label"], fontsize=17, color=CONFIG_COLORS[row["config"]], va="center",
    )

ax.set_xlabel("Carbon Footprint Decrease (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Traffic (%)", fontsize=LABEL_SIZE, labelpad=12)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlim(-4, 96)
max_dirty_grid_share = max(
    (
        high_carbon_summary_df[f"traffic_ge_{threshold}_share"]
        + high_carbon_summary_df[f"traffic_ge_{threshold}_share_std"]
    ).max()
    for threshold in dirty_thresholds
)
ax.set_ylim(0, max(26, max_dirty_grid_share + 3))
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.legend(ncol=1, loc="upper right", frameon=True, fontsize=16)
format_axes(ax)
fig.subplots_adjust(left=0.18, right=0.97, bottom=0.18, top=0.96)

out_path = FIGURES_DIR / "dirty_grid_traffic_vs_carbon_saved.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()


## Load security data

In [ ]:
# One row per network, configuration, attacker, and attacker budget.
SECURITY_ATTACKERS = {
    "random": ("cache_sybil_random.csv", "Naive: random relays"),
    "low_ci": ("cache_sybil_ci.csv", "Adaptive: lowest-carbon relays"),
    "high_bw": ("cache_sybil_bw.csv", "Highest-bandwidth relays"),
}

security_sybil_parts = []
security_sybil_missing = []

for network in NETWORKS:
    for config in CONFIG_ORDER:
        config_dir = config_paths[network][config]
        for strategy, (filename, attacker_label) in SECURITY_ATTACKERS.items():
            cache_path = config_dir / filename
            if not cache_path.exists():
                security_sybil_missing.append({
                    "network": network,
                    "config": config,
                    "strategy": strategy,
                    "missing": filename,
                })
                continue

            sybil = pd.read_csv(cache_path)
            sybil["network"] = network
            sybil["config"] = config
            sybil["config_label"] = CONFIG_LABELS[config]
            sybil["strategy"] = strategy
            sybil["attacker_label"] = attacker_label
            security_sybil_parts.append(sybil)

security_sybil_df = pd.concat(security_sybil_parts, ignore_index=True)
security_sybil_df["percent"] = pd.to_numeric(security_sybil_df["percent"])
security_sybil_missing_df = pd.DataFrame(security_sybil_missing)

security_sybil_coverage_df = (
    security_sybil_df.groupby(["network", "config", "strategy"], as_index=False)
    .agg(attacker_budgets=("percent", "nunique"), circuits=("total_circuits", "max"))
)

print(f"Sybil rows: {len(security_sybil_df):,}")
print(f"Missing Sybil caches: {len(security_sybil_missing_df):,}")
security_sybil_coverage_df.head()

In [ ]:
# Aggregate the hourly ASN caches within each network and configuration.
security_asn_network_rows = []
security_asn_match_rows = []

for network in NETWORKS:
    for config in CONFIG_ORDER:
        total_circuits = 0
        hours = 0
        guard_exit_by_asn = {}
        all_three_by_asn = {}

        for hour_dir in hour_dirs_for(network, config_paths[network][config]):
            cache_path = hour_dir / "cache_asn_analysis.json"
            if not cache_path.exists():
                continue

            cached = json.loads(cache_path.read_text())
            total_circuits += cached["total"]
            hours += 1

            for asn, count in cached["match_1_3"].items():
                guard_exit_by_asn[asn] = guard_exit_by_asn.get(asn, 0) + count
            for asn, count in cached["match_all"].items():
                all_three_by_asn[asn] = all_three_by_asn.get(asn, 0) + count

        guard_exit_matches = sum(guard_exit_by_asn.values())
        all_three_matches = sum(all_three_by_asn.values())
        security_asn_network_rows.append({
            "network": network,
            "config": config,
            "config_label": CONFIG_LABELS[config],
            "hours": hours,
            "total_circuits": total_circuits,
            "guard_exit_matches": guard_exit_matches,
            "all_three_matches": all_three_matches,
            "guard_exit_pct": guard_exit_matches / total_circuits * 100,
            "all_three_pct": all_three_matches / total_circuits * 100,
        })

        for outcome, matches in [
            ("Guard and exit", guard_exit_by_asn),
            ("All three", all_three_by_asn),
        ]:
            for asn, count in matches.items():
                security_asn_match_rows.append({
                    "network": network,
                    "config": config,
                    "config_label": CONFIG_LABELS[config],
                    "outcome": outcome,
                    "asn": asn,
                    "matches": count,
                    "total_circuits": total_circuits,
                })

security_asn_network_df = pd.DataFrame(security_asn_network_rows)
security_asn_match_df = pd.DataFrame(security_asn_match_rows)

print(f"ASN network/config rows: {len(security_asn_network_df):,}")
security_asn_network_df.head()

## Malicious relays

The random attacker is unaware of the selection policy. The lowest-carbon attacker targets the relays favored by carbon-aware selection, while the highest-bandwidth attacker targets the relays favored by Tor's baseline. The main panels show the complete 1--100% range; the insets use independent y-axis scales to show 1--10%.

In [ ]:
# Mean and one standard deviation across networks.
security_sybil_summary_df = (
    security_sybil_df.groupby(
        ["strategy", "attacker_label", "config", "config_label", "percent"],
        as_index=False,
    )
    .agg(
        networks=("network", "nunique"),
        guard_exit_mean=("match_rate_2hop", "mean"),
        guard_exit_std=("match_rate_2hop", "std"),
        all_three_mean=("match_rate_3hop", "mean"),
        all_three_std=("match_rate_3hop", "std"),
    )
)

for col in ["guard_exit_mean", "guard_exit_std", "all_three_mean", "all_three_std"]:
    security_sybil_summary_df[col] *= 100

security_sybil_10pct_df = security_sybil_summary_df[
    security_sybil_summary_df["percent"] == 10
].copy()

for metric in ["guard_exit", "all_three"]:
    baseline = (
        security_sybil_10pct_df[security_sybil_10pct_df["config"] == "baseline"]
        .set_index("strategy")[f"{metric}_mean"]
    )
    security_sybil_10pct_df[f"{metric}_vs_baseline"] = (
        security_sybil_10pct_df.apply(
            lambda row: row[f"{metric}_mean"] / baseline[row["strategy"]], axis=1
        )
    )

security_sybil_10pct_df["config_rank"] = security_sybil_10pct_df["config"].map(config_rank)
security_sybil_10pct_df["strategy_rank"] = security_sybil_10pct_df["strategy"].map(
    {"random": 0, "low_ci": 1, "high_bw": 2}
)
security_sybil_10pct_df = security_sybil_10pct_df.sort_values(
    ["strategy_rank", "config_rank"]
)

security_sybil_10pct_display_df = pd.DataFrame({
    "Attacker": security_sybil_10pct_df["attacker_label"],
    "Setting": security_sybil_10pct_df["config_label"],
    "Guard + exit": security_sybil_10pct_df["guard_exit_mean"].map(lambda v: f"{v:.2f}%"),
    "G+E / Base": security_sybil_10pct_df["guard_exit_vs_baseline"].map(lambda v: f"{v:.2f}x"),
    "All three": security_sybil_10pct_df["all_three_mean"].map(lambda v: f"{v:.2f}%"),
    "All 3 / Base": security_sybil_10pct_df["all_three_vs_baseline"].map(lambda v: f"{v:.2f}x"),
})

security_sybil_csv_path = FIGURES_DIR / "security_sybil_summary.csv"
security_sybil_10pct_csv_path = FIGURES_DIR / "security_sybil_10pct_summary.csv"
security_sybil_summary_df.to_csv(security_sybil_csv_path, index=False)
security_sybil_10pct_display_df.to_csv(security_sybil_10pct_csv_path, index=False)
print(security_sybil_csv_path)
print(security_sybil_10pct_csv_path)
security_sybil_10pct_display_df

In [ ]:
LABEL_SIZE_LARGE = LABEL_SIZE + 6

In [ ]:
# Guard and exit compromise rate. This is the main relay-adversary figure.
strategy_order = ["random", "low_ci", "high_bw"]
strategy_titles = {
    "random": "Attacker Controls Random Relays",
    "low_ci": "Attacker Controls\nLowest-to-Highest Carbon Relays",
    "high_bw": "Attacker Controls\nHighest-to-Lowest Bandwidth Relays",
}

fig, axes = plt.subplots(1, 3, figsize=(18.0, 7.0), sharey=True)

for ax, strategy in zip(axes, strategy_order):
    strategy_df = security_sybil_summary_df[
        security_sybil_summary_df["strategy"] == strategy
    ]
    if strategy == "random":
        axins = ax.inset_axes([0.19, 0.62, 0.50, 0.35])

    for config in CONFIG_ORDER:
        sub = strategy_df[strategy_df["config"] == config].sort_values("percent")
        lower = np.maximum(0, sub["guard_exit_mean"] - sub["guard_exit_std"])
        upper = sub["guard_exit_mean"] + sub["guard_exit_std"]
        ax.plot(
            sub["percent"], sub["guard_exit_mean"],
            color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
            linewidth=LINE_WIDTH, markersize=MARKER_SIZE, markevery=10,
            label=CONFIG_LABELS[config],
        )
        ax.fill_between(sub["percent"], lower, upper, color=CONFIG_COLORS[config], alpha=0.13)

        if strategy == "random":
            zoom = sub[sub["percent"] <= 10]
            zoom_lower = np.maximum(0, zoom["guard_exit_mean"] - zoom["guard_exit_std"])
            zoom_upper = zoom["guard_exit_mean"] + zoom["guard_exit_std"]
            axins.plot(
                zoom["percent"], zoom["guard_exit_mean"],
                color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
                linewidth=2.2, markersize=5,
            )
            axins.fill_between(
                zoom["percent"], zoom_lower, zoom_upper,
                color=CONFIG_COLORS[config], alpha=0.10,
            )

    ax.set_title(strategy_titles[strategy], fontsize=FONT_SIZE, pad=12)
    if strategy == "low_ci":
        ax.set_xlabel("Attacker-controlled relays (%)", fontsize=LABEL_SIZE_LARGE, labelpad=10)
    else: 
        ax.set_xlabel("")
    ax.set_xlim(1, 100)
    ax.set_ylim(0, 102)
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_yticks([1, 25, 50, 75, 100])
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis="both", linestyle="--", alpha=0.25)
    ax.tick_params(axis="both", labelsize=LABEL_SIZE_LARGE+1)
    format_axes(ax)
    if ax is not axes[0]:
        plt.setp(ax.get_yticklabels(), visible=False)

    if strategy == "random":
        zoom_df = strategy_df[strategy_df["percent"] <= 10]
        zoom_max = (zoom_df["guard_exit_mean"] + zoom_df["guard_exit_std"]).max()
        axins.set_xlim(1, 10)
        axins.set_ylim(0, max(0.1, zoom_max * 1.05))
        axins.set_xticks([1, 5, 10])
        axins.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
        axins.yaxis.set_major_locator(mtick.MaxNLocator(nbins=3))
        axins.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=1))
        axins.set_facecolor("white")
        axins.text(
            0.50, 0.96, "1-10% zoom", transform=axins.transAxes,
            ha="center", va="top", fontsize=18,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 1.5},
        )
        axins.grid(axis="both", linestyle="--", alpha=0.20)
        axins.tick_params(axis="both", labelsize=20)
        format_axes(axins)

axes[0].set_ylabel("Guard + Exit Comprom. (%)", fontsize=LABEL_SIZE_LARGE, labelpad=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="upper center", frameon=True, fontsize=LABEL_SIZE_LARGE)
fig.subplots_adjust(left=0.09, right=0.99, bottom=0.20, top=0.72, wspace=0.30)

out_path = FIGURES_DIR / "security_sybil_guard_exit_all_networks.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

In [ ]:
# All-three compromise rate, separated because it is much smaller.
fig, axes = plt.subplots(1, 3, figsize=(18.0, 7.0), sharey=True)

for ax, strategy in zip(axes, strategy_order):
    strategy_df = security_sybil_summary_df[
        security_sybil_summary_df["strategy"] == strategy
    ]
    if strategy == "random":
        axins = ax.inset_axes([0.19, 0.62, 0.50, 0.35])

    for config in CONFIG_ORDER:
        sub = strategy_df[strategy_df["config"] == config].sort_values("percent")
        lower = np.maximum(0, sub["all_three_mean"] - sub["all_three_std"])
        upper = sub["all_three_mean"] + sub["all_three_std"]
        ax.plot(
            sub["percent"], sub["all_three_mean"],
            color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
            linewidth=LINE_WIDTH, markersize=MARKER_SIZE, markevery=10,
            label=CONFIG_LABELS[config],
        )
        ax.fill_between(sub["percent"], lower, upper, color=CONFIG_COLORS[config], alpha=0.13)

        if strategy == "random":
            zoom = sub[sub["percent"] <= 10]
            zoom_lower = np.maximum(0, zoom["all_three_mean"] - zoom["all_three_std"])
            zoom_upper = zoom["all_three_mean"] + zoom["all_three_std"]
            axins.plot(
                zoom["percent"], zoom["all_three_mean"],
                color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
                linewidth=2.2, markersize=5,
            )
            axins.fill_between(
                zoom["percent"], zoom_lower, zoom_upper,
                color=CONFIG_COLORS[config], alpha=0.10,
            )

    ax.set_title(strategy_titles[strategy], fontsize=FONT_SIZE, pad=12)
    if strategy == "low_ci":
        ax.set_xlabel("Attacker-controlled relays (%)", fontsize=LABEL_SIZE_LARGE, labelpad=10)
    else:
        ax.set_xlabel("")
    ax.set_xlim(1, 100)
    ax.set_ylim(0, 102)
    ax.set_xticks([1, 25, 50, 75, 100])
    ax.set_yticks([1, 25, 50, 75, 100])
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
    ax.grid(axis="both", linestyle="--", alpha=0.25)
    ax.tick_params(axis="both", labelsize=LABEL_SIZE_LARGE+1)
    format_axes(ax)
    if ax is not axes[0]:
        plt.setp(ax.get_yticklabels(), visible=False)

    if strategy == "random":
        zoom_df = strategy_df[strategy_df["percent"] <= 10]
        zoom_max = (zoom_df["all_three_mean"] + zoom_df["all_three_std"]).max()
        axins.set_xlim(1, 10)
        axins.set_ylim(0, max(0.01, zoom_max * 1.05))
        axins.set_xticks([1, 5, 10])
        axins.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
        axins.yaxis.set_major_locator(mtick.MaxNLocator(nbins=3))
        axins.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=2))
        axins.set_facecolor("white")
        axins.text(
            0.50, 0.96, "1-10% zoom", transform=axins.transAxes,
            ha="center", va="top", fontsize=18,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 1.5},
        )
        axins.grid(axis="both", linestyle="--", alpha=0.20)
        axins.tick_params(axis="both", labelsize=20)
        format_axes(axins)

axes[0].set_ylabel("All 3 Relays Comprom. (%)", fontsize=LABEL_SIZE_LARGE, labelpad=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=6, loc="upper center", frameon=True, fontsize=LABEL_SIZE_LARGE)
fig.subplots_adjust(left=0.09, right=0.99, bottom=0.20, top=0.72, wspace=0.30)

out_path = FIGURES_DIR / "security_sybil_all_three_all_networks.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

## Autonomous systems

The cached ASN analysis checks whether selected relays share the same hosting ASN. In the figure below, white points are individual generated networks, bars are the unweighted mean across networks, and whiskers show one standard deviation. This is a relay co-location measure, not a full AS-path correlation analysis: it does not include ASes on the client-to-guard or exit-to-destination paths, route asymmetry, or BGP manipulation.

In [ ]:
# ASN co-location summary across networks.
security_asn_summary_df = (
    security_asn_network_df.groupby(["config", "config_label"], as_index=False)
    .agg(
        networks=("network", "nunique"),
        hours_min=("hours", "min"),
        guard_exit_mean=("guard_exit_pct", "mean"),
        guard_exit_std=("guard_exit_pct", "std"),
        all_three_mean=("all_three_pct", "mean"),
        all_three_std=("all_three_pct", "std"),
    )
)
security_asn_summary_df["config_rank"] = security_asn_summary_df["config"].map(config_rank)
security_asn_summary_df = security_asn_summary_df.sort_values("config_rank").drop(columns="config_rank")

security_asn_display_df = pd.DataFrame({
    "Setting": security_asn_summary_df["config_label"],
    "Guard + exit": security_asn_summary_df.apply(
        lambda row: f"{row['guard_exit_mean']:.3f}% +/- {row['guard_exit_std']:.3f}", axis=1
    ),
    "All three": security_asn_summary_df.apply(
        lambda row: f"{row['all_three_mean']:.3f}% +/- {row['all_three_std']:.3f}", axis=1
    ),
})

security_asn_contributors_df = (
    security_asn_match_df.groupby(["outcome", "asn"], as_index=False)
    .agg(matches=("matches", "sum"))
)
security_asn_contributors_df["matched_share_pct"] = (
    security_asn_contributors_df["matches"]
    / security_asn_contributors_df.groupby("outcome")["matches"].transform("sum") * 100
)
security_asn_top_df = (
    security_asn_contributors_df.sort_values(["outcome", "matches"], ascending=[True, False])
    .groupby("outcome", as_index=False)
    .head(5)
)

security_asn_csv_path = FIGURES_DIR / "security_asn_summary.csv"
security_asn_top_csv_path = FIGURES_DIR / "security_asn_top_contributors.csv"
security_asn_summary_df.to_csv(security_asn_csv_path, index=False)
security_asn_top_df.to_csv(security_asn_top_csv_path, index=False)
print(security_asn_csv_path)
print(security_asn_top_csv_path)
security_asn_display_df

In [ ]:
# Bars show the network mean and one standard deviation; dots are individual networks.
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.8))
asn_plot_specs = [
    ("guard_exit_mean", "guard_exit_std", "guard_exit_pct", "Guard and exit relays share an ASN"),
    ("all_three_mean", "all_three_std", "all_three_pct", "All three relays share an ASN"),
]
x = np.arange(len(CONFIG_ORDER))
network_offsets = dict(zip(NETWORKS, np.linspace(-0.14, 0.14, len(NETWORKS))))

for ax, (mean_col, std_col, network_col, title) in zip(axes, asn_plot_specs):
    means = security_asn_summary_df.set_index("config").loc[CONFIG_ORDER, mean_col]
    errors = security_asn_summary_df.set_index("config").loc[CONFIG_ORDER, std_col]
    ax.bar(
        x, means, yerr=errors,
        color=[CONFIG_COLORS[config] for config in CONFIG_ORDER],
        edgecolor="black", linewidth=0.8, alpha=0.78,
        error_kw={"elinewidth": 2.0, "capsize": 5, "capthick": 2.0},
    )

    for config_i, config in enumerate(CONFIG_ORDER):
        points = security_asn_network_df[security_asn_network_df["config"] == config]
        for _, row in points.iterrows():
            ax.scatter(
                config_i + network_offsets[row["network"]], row[network_col],
                s=55, color="white", edgecolor="black", linewidth=1.2, zorder=3,
            )

    ax.set_title(title, fontsize=FONT_SIZE, pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(["Base.", "T90", "T85", "T80", "T75", "T70"])
    ax.set_ylabel("Circuits (%)", fontsize=LABEL_SIZE, labelpad=10)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=2))
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.tick_params(axis="both", labelsize=LABEL_SIZE)
    format_axes(ax)
print(FONT_SIZE)
print(LABEL_SIZE)
print(LABEL_SIZE_LARGE)
fig.subplots_adjust(left=0.10, right=0.99, bottom=0.17, top=0.86, wspace=0.34)
out_path = FIGURES_DIR / "security_asn_correlation_all_networks.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

### ASNs contributing to relay co-location

We report public ASN identifiers only and describe them as contributors to same-AS circuits. Hosting multiple selected Tor relays is not evidence that an AS is malicious, controls those relays, or performs traffic correlation.

In [ ]:
# Pooled guard-and-exit co-location rate for the most frequent ASNs.
guard_exit_asn_df = (
    security_asn_match_df[security_asn_match_df["outcome"] == "Guard and exit"]
    .groupby(["config", "config_label", "asn"], as_index=False)
    .agg(matches=("matches", "sum"))
)
asn_config_totals_df = (
    security_asn_network_df.groupby(["config", "config_label"], as_index=False)
    .agg(total_circuits=("total_circuits", "sum"))
)
guard_exit_asn_df = guard_exit_asn_df.merge(
    asn_config_totals_df, on=["config", "config_label"], how="left"
)
guard_exit_asn_df["circuit_pct"] = (
    guard_exit_asn_df["matches"] / guard_exit_asn_df["total_circuits"] * 100
)

top_guard_exit_asns = (
    guard_exit_asn_df.groupby("asn")["matches"].sum()
    .nlargest(6)
    .index.tolist()
)
guard_exit_top_df = guard_exit_asn_df[guard_exit_asn_df["asn"].isin(top_guard_exit_asns)].copy()
guard_exit_asn_matrix_df = (
    guard_exit_top_df.pivot(index="config", columns="asn", values="circuit_pct")
    .reindex(index=CONFIG_ORDER, columns=top_guard_exit_asns)
    .fillna(0)
)

guard_exit_total_pct = (
    security_asn_network_df.groupby("config")
    .apply(lambda sub: sub["guard_exit_matches"].sum() / sub["total_circuits"].sum() * 100)
    .reindex(CONFIG_ORDER)
)
guard_exit_asn_matrix_df["Other"] = (
    guard_exit_total_pct - guard_exit_asn_matrix_df.sum(axis=1)
).clip(lower=0)

guard_exit_asn_heatmap_csv_path = FIGURES_DIR / "security_asn_guard_exit_contributors.csv"
guard_exit_asn_matrix_df.to_csv(guard_exit_asn_heatmap_csv_path)
print(guard_exit_asn_heatmap_csv_path)
guard_exit_asn_matrix_df

In [ ]:
# Heatmap of each ASN's contribution to guard-and-exit same-AS circuits.
fig, ax = plt.subplots(figsize=(14.0, 6.0))
heatmap_values = guard_exit_asn_matrix_df.to_numpy()
image = ax.imshow(heatmap_values, cmap="YlOrRd", aspect="auto", vmin=0)

ax.set_xticks(np.arange(len(guard_exit_asn_matrix_df.columns)))
ax.set_xticklabels(
    [f"ASN {asn}" if asn != "Other" else "Other" for asn in guard_exit_asn_matrix_df.columns],
    rotation=25, ha="right",
)
ax.set_yticks(np.arange(len(CONFIG_ORDER)))
ax.set_yticklabels(["Base.", "T90", "T85", "T80", "T75", "T70"])
ax.set_xlabel("Relay hosting ASN", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Configuration", fontsize=LABEL_SIZE, labelpad=10)
ax.tick_params(axis="both", labelsize=LABEL_SIZE_LARGE)

threshold = heatmap_values.max() * 0.55
for row in range(heatmap_values.shape[0]):
    for col in range(heatmap_values.shape[1]):
        value = heatmap_values[row, col]
        ax.text(
            col, row, f"{value:.3f}%",
            ha="center", va="center", fontsize=LABEL_SIZE,
            color="white" if value >= threshold else "black",
        )

colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("All circuits (%)", fontsize=LABEL_SIZE, labelpad=10)
colorbar.ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=2))
colorbar.ax.tick_params(labelsize=FONT_SIZE)
format_axes(ax)
fig.subplots_adjust(left=0.12, right=0.95, bottom=0.28, top=0.97)

out_path = FIGURES_DIR / "security_asn_guard_exit_contributors.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

## Additional security models

In [ ]:
# Load the relay fields used by the additional security models.
security_relay_columns = [
    "relay", "country", "circuit_count", "circuits_built", "throughput",
    "carbon_intensity", "bw_rate", "error_count", "write_congested_s",
]
security_relay_hour_parts = []

for network in NETWORKS:
    for config in CONFIG_ORDER:
        for hour_dir in hour_dirs_for(network, config_paths[network][config]):
            emissions_path = hour_dir / "cache_emissions.csv"
            if not emissions_path.exists():
                continue

            available_columns = pd.read_csv(emissions_path, nrows=0).columns
            usecols = [column for column in security_relay_columns if column in available_columns]
            relay_hour = pd.read_csv(emissions_path, usecols=usecols)
            for column in security_relay_columns:
                if column not in relay_hour.columns:
                    relay_hour[column] = 0

            relay_hour["relay"] = relay_hour["relay"].fillna("Unknown").astype(str)
            relay_hour["country"] = relay_hour["country"].fillna("Unknown").astype(str)
            relay_hour["network"] = network
            relay_hour["config"] = config
            relay_hour["config_label"] = CONFIG_LABELS[config]
            security_relay_hour_parts.append(relay_hour)

security_relay_hour_df = pd.concat(security_relay_hour_parts, ignore_index=True)
for column in security_relay_columns[2:]:
    security_relay_hour_df[column] = pd.to_numeric(
        security_relay_hour_df[column], errors="coerce"
    ).fillna(0)

# One row per relay, network, and configuration across the observed hours.
security_relay_daily_df = (
    security_relay_hour_df.groupby(
        ["network", "config", "config_label", "relay", "country"],
        as_index=False,
    )
    .agg(
        circuit_uses=("circuit_count", "sum"),
        circuits_built=("circuits_built", "sum"),
        throughput_bytes=("throughput", "sum"),
        carbon_intensity=("carbon_intensity", "mean"),
        bw_rate=("bw_rate", "max"),
        error_count=("error_count", "sum"),
        write_congested_s=("write_congested_s", "sum"),
    )
)
security_relay_daily_df["eligible_guard"] = security_relay_daily_df["relay"].str.contains("guard")
security_relay_daily_df["eligible_exit"] = security_relay_daily_df["relay"].str.contains("exit")
security_relay_daily_df["relay_role"] = np.select(
    [
        security_relay_daily_df["eligible_guard"] & security_relay_daily_df["eligible_exit"],
        security_relay_daily_df["eligible_guard"],
        security_relay_daily_df["eligible_exit"],
    ],
    ["Guard + Exit", "Guard", "Exit"],
    default="Middle",
)
security_relay_daily_df["load_share"] = (
    security_relay_daily_df["circuit_uses"]
    / security_relay_daily_df.groupby(["network", "config"])["circuit_uses"].transform("sum")
)

print(f"Security relay rows: {len(security_relay_daily_df):,}")
security_relay_daily_df.head()

### 1. Position-coupled attacker exposure

The Sybil attacks retain end-to-end compromise but not individual circuit positions. It would be good to pair the measured compromise rate with the fraction of load carried by guard-eligible and exit-eligible relays selected by each attacker This follows the resource-amplification question raised by [low-resource routing attacks](https://damonmccoy.com/papers/wpes25-bauer.pdf).

In [ ]:
# Exposure of role-eligible relay load to an attacker controlling 10% of relays.
ATTACK_BUDGET_PCT = 10
SECURITY_STRATEGY_LABELS = {
    "random": "Random",
    "low_ci": "Lowest carbon",
    "high_bw": "Highest bandwidth",
}
SECURITY_STRATEGY_COLORS = {
    "random": "#3f3f3f",
    "low_ci": "#2ca02c",
    "high_bw": "#d62728",
}
SECURITY_STRATEGY_MARKERS = {"random": "o", "low_ci": "^", "high_bw": "s"}

position_exposure_rows = []
for (network, config), relays in security_relay_daily_df.groupby(["network", "config"]):
    relays = relays.copy()
    attacker_relays = max(1, int(np.ceil(len(relays) * ATTACK_BUDGET_PCT / 100)))

    for strategy in SECURITY_STRATEGY_LABELS:
        if strategy == "low_ci":
            selected = relays.sort_values(
                ["carbon_intensity", "circuit_uses"], ascending=[True, False]
            ).head(attacker_relays)
        elif strategy == "high_bw":
            selected = relays.sort_values(
                ["bw_rate", "circuit_uses"], ascending=False
            ).head(attacker_relays)
        else:
            selected = None

        for role, role_column in [("Guard-eligible", "eligible_guard"), ("Exit-eligible", "eligible_exit")]:
            role_relays = relays[relays[role_column]]
            role_load = role_relays["circuit_uses"].sum()
            if strategy == "random":
                exposure_pct = attacker_relays / len(relays) * 100
            else:
                controlled_load = selected[selected[role_column]]["circuit_uses"].sum()
                exposure_pct = controlled_load / role_load * 100 if role_load else np.nan

            position_exposure_rows.append({
                "network": network,
                "config": config,
                "strategy": strategy,
                "role": role,
                "exposure_pct": exposure_pct,
            })

position_exposure_network_df = pd.DataFrame(position_exposure_rows)
position_exposure_summary_df = (
    position_exposure_network_df.groupby(["config", "strategy", "role"], as_index=False)
    .agg(exposure_pct=("exposure_pct", "mean"), exposure_std=("exposure_pct", "std"))
)

compromise_amplification_10_df = security_sybil_summary_df[
    security_sybil_summary_df["percent"] == ATTACK_BUDGET_PCT
].copy()
uniform_guard_exit_pct = ATTACK_BUDGET_PCT ** 2 / 100
compromise_amplification_10_df["amplification"] = (
    compromise_amplification_10_df["guard_exit_mean"] / uniform_guard_exit_pct
)
compromise_amplification_10_df["amplification_std"] = (
    compromise_amplification_10_df["guard_exit_std"] / uniform_guard_exit_pct
)

position_exposure_summary_df.to_csv(
    FIGURES_DIR / "security_position_exposure_10pct.csv", index=False
)
compromise_amplification_10_df.to_csv(
    FIGURES_DIR / "security_compromise_amplification_10pct.csv", index=False
)
position_exposure_summary_df.head()

### 2. Jurisdictional concentration

Following the country-level adversary in [TAPS](https://www.ndss-symposium.org/ndss2017/ndss-2017-programme/avoding-man-wire-improving-tors-security-trust-aware-path-selection/), this measures how much simulated relay traffic the largest country coalitions cover. It is a traffic-concentration proxy; exact guard–exit jurisdiction compromise requires retained circuit paths.

In [ ]:
# Cumulative relay-traffic coverage by the largest country coalitions.
jurisdiction_rows = []
jurisdiction_threshold_rows = []

for (network, config), countries in country_daily_load_df.groupby(["network", "config"]):
    countries = countries.sort_values("share", ascending=False).reset_index(drop=True)
    cumulative = countries["share"].cumsum() * 100
    for coalition_size in range(1, min(15, len(countries)) + 1):
        jurisdiction_rows.append({
            "network": network,
            "config": config,
            "coalition_size": coalition_size,
            "traffic_coverage_pct": cumulative.iloc[coalition_size - 1],
        })
    for threshold in [25, 50, 75]:
        reached = np.flatnonzero(cumulative.to_numpy() >= threshold)
        jurisdiction_threshold_rows.append({
            "network": network,
            "config": config,
            "threshold_pct": threshold,
            "countries_needed": int(reached[0] + 1) if len(reached) else np.nan,
        })

jurisdiction_network_df = pd.DataFrame(jurisdiction_rows)
jurisdiction_summary_df = (
    jurisdiction_network_df.groupby(["config", "coalition_size"], as_index=False)
    .agg(
        traffic_coverage_pct=("traffic_coverage_pct", "mean"),
        traffic_coverage_std=("traffic_coverage_pct", "std"),
    )
)
jurisdiction_threshold_network_df = pd.DataFrame(jurisdiction_threshold_rows)
jurisdiction_threshold_summary_df = (
    jurisdiction_threshold_network_df.groupby(["config", "threshold_pct"], as_index=False)
    .agg(countries_needed=("countries_needed", "mean"), countries_needed_std=("countries_needed", "std"))
)

jurisdiction_summary_df.to_csv(FIGURES_DIR / "security_jurisdiction_coalitions.csv", index=False)
jurisdiction_threshold_summary_df.to_csv(
    FIGURES_DIR / "security_jurisdiction_thresholds.csv", index=False
)
jurisdiction_threshold_summary_df

### 3. Persistent guard exposure

[Tor's guard design](https://spec.torproject.org/guard-spec/) keeps a small guard set rather than choosing a fresh guard independently for every circuit. Without multi-month client state in the retained logs, we model repeated independent samples of three guards using the measured guard-eligible load exposure above. This is a sensitivity analysis, not a timeline.

In [ ]:
# Probability that at least one attacker guard is sampled across repeated guard-set draws.
persistent_guard_rows = []
guard_exposure_network_df = position_exposure_network_df[
    position_exposure_network_df["role"] == "Guard-eligible"
].copy()

for row in guard_exposure_network_df.itertuples(index=False):
    attacker_guard_probability = np.clip(row.exposure_pct / 100, 0, 1)
    for guard_set_samples in range(1, 13):
        sampled_guards = 3 * guard_set_samples
        persistent_guard_rows.append({
            "network": row.network,
            "config": row.config,
            "strategy": row.strategy,
            "guard_set_samples": guard_set_samples,
            "sampled_guards": sampled_guards,
            "attacker_guard_probability_pct": (
                1 - (1 - attacker_guard_probability) ** sampled_guards
            ) * 100,
        })

persistent_guard_network_df = pd.DataFrame(persistent_guard_rows)
persistent_guard_summary_df = (
    persistent_guard_network_df.groupby(
        ["config", "strategy", "guard_set_samples", "sampled_guards"], as_index=False
    )
    .agg(
        attacker_guard_probability_pct=("attacker_guard_probability_pct", "mean"),
        attacker_guard_probability_std=("attacker_guard_probability_pct", "std"),
    )
)
persistent_guard_summary_df.to_csv(
    FIGURES_DIR / "security_persistent_guard_exposure.csv", index=False
)
persistent_guard_summary_df.head()

In [ ]:
# Repeated three-guard samples for a random-relay attacker.
fig, ax = plt.subplots(figsize=(8, 4.5))

for config in CONFIG_ORDER:
    sub = persistent_guard_summary_df[
        (persistent_guard_summary_df["strategy"] == "random")
        & (persistent_guard_summary_df["config"] == config)
    ].sort_values("guard_set_samples")
    lower = np.maximum(
        0,
        sub["attacker_guard_probability_pct"]
        - sub["attacker_guard_probability_std"].fillna(0),
    )
    upper = np.minimum(
        100,
        sub["attacker_guard_probability_pct"]
        + sub["attacker_guard_probability_std"].fillna(0),
    )
    ax.plot(
        sub["guard_set_samples"], sub["attacker_guard_probability_pct"],
        color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
        linewidth=LINE_WIDTH, markersize=MARKER_SIZE, markevery=2,
        label=CONFIG_LABELS[config],
    )
    ax.fill_between(
        sub["guard_set_samples"], lower, upper,
        color=CONFIG_COLORS[config], alpha=0.12,
    )

ax.set_xlabel("Independent three-guard samples", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("At least one attacker\nguard selected (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_xlim(1, 12)
ax.set_ylim(0, 102)
ax.set_xticks([1, 3, 6, 9, 12])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.grid(axis="both", linestyle="--", alpha=0.25)
format_axes(ax)
ax.legend(ncol=1, loc="lower right", fontsize=LEGEND_SIZE, frameon=True)
fig.subplots_adjust(left=0.15, right=0.99, bottom=0.19, top=0.86)

out_path = FIGURES_DIR / "security_persistent_guard_exposure_random.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

The path is a TODO for next time

In [ ]:
# Lower bound from coalitions of relay-hosting ASNs with same-AS Guard + Exit circuits.
guard_exit_provider_df = security_asn_match_df[
    security_asn_match_df["outcome"] == "Guard and exit"
].copy()

hosting_as_coalition_rows = []

for (network, config), matches in guard_exit_provider_df.groupby(["network", "config"]):
    asn_counts = matches.groupby("asn")["matches"].sum().sort_values(ascending=False)
    total_circuits = matches["total_circuits"].iloc[0]
    cumulative_matches = asn_counts.cumsum()
    for coalition_size in range(1, min(10, len(cumulative_matches)) + 1):
        hosting_as_coalition_rows.append({
            "network": network,
            "config": config,
            "coalition_size": coalition_size,
            "same_host_as_circuit_pct": (
                cumulative_matches.iloc[coalition_size - 1] / total_circuits * 100
            ),
        })

hosting_as_coalition_network_df = pd.DataFrame(hosting_as_coalition_rows)
hosting_as_coalition_summary_df = (
    hosting_as_coalition_network_df.groupby(["config", "coalition_size"], as_index=False)
    .agg(
        same_host_as_circuit_pct=("same_host_as_circuit_pct", "mean"),
        same_host_as_circuit_std=("same_host_as_circuit_pct", "std"),
    )
)
hosting_as_coalition_summary_df.to_csv(
    FIGURES_DIR / "security_hosting_as_coalition_lower_bound.csv", index=False
)
hosting_as_coalition_summary_df.head()

In [ ]:
# Lower-bound circuit exposure to coalitions of relay-hosting ASNs.
fig, ax = plt.subplots(figsize=(10.5, 6.5))

for config in CONFIG_ORDER:
    sub = hosting_as_coalition_summary_df[
        hosting_as_coalition_summary_df["config"] == config
    ].sort_values("coalition_size")
    lower = np.maximum(
        0, sub["same_host_as_circuit_pct"] - sub["same_host_as_circuit_std"].fillna(0)
    )
    upper = sub["same_host_as_circuit_pct"] + sub["same_host_as_circuit_std"].fillna(0)
    ax.plot(
        sub["coalition_size"], sub["same_host_as_circuit_pct"],
        color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
        linewidth=LINE_WIDTH, markersize=MARKER_SIZE,
        label=CONFIG_LABELS[config],
    )
    ax.fill_between(
        sub["coalition_size"], lower, upper,
        color=CONFIG_COLORS[config], alpha=0.12,
    )

ax.set_xlabel("Largest relay-hosting ASN coalition", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Same-host-AS circuits (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=1))
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.legend(ncol=3, fontsize=LEGEND_SIZE, frameon=True, loc="upper left")
format_axes(ax)
fig.subplots_adjust(left=0.16, right=0.98, bottom=0.18, top=0.97)

out_path = FIGURES_DIR / "security_hosting_as_coalition_lower_bound.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

### 6. Targeted relay outage

Motivated by targeted relay attacks such as [Sniper](https://www.robgjansen.com/publications/sniper-ndss2014.html) and [Point Break](https://www.usenix.org/system/files/sec19-jansen.pdf), this first-order stress test removes either the most-used relays or the lowest-carbon relays and reports their observed share of relay selections. It measures immediate exposure before path rebuilding. 

In [ ]:
# Estimated all-three circuit exposure to the largest colluding-country sets.
country_collusion_rows = []

for (network, config), relays in security_relay_daily_df.groupby(["network", "config"]):
    countries = (
        relays.groupby("country", as_index=False)
        .agg(circuit_uses=("circuit_uses", "sum"))
        .sort_values("circuit_uses", ascending=False)
        .reset_index(drop=True)
    )
    countries["selection_share"] = countries["circuit_uses"] / countries["circuit_uses"].sum()
    cumulative_share = countries["selection_share"].cumsum()

    for colluding_countries in range(1, min(10, len(countries)) + 1):
        colluding_selection_share = cumulative_share.iloc[colluding_countries - 1]
        country_collusion_rows.append({
            "network": network,
            "config": config,
            "colluding_countries": colluding_countries,
            "colluding_selection_share": colluding_selection_share,
            "all_three_circuits_pct": colluding_selection_share ** 3 * 100,
        })

country_collusion_network_df = pd.DataFrame(country_collusion_rows)
country_collusion_summary_df = (
    country_collusion_network_df.groupby(["config", "colluding_countries"], as_index=False)
    .agg(
        all_three_circuits_pct=("all_three_circuits_pct", "mean"),
        all_three_circuits_std=("all_three_circuits_pct", "std"),
    )
)

country_collusion_summary_df.to_csv(
    FIGURES_DIR / "security_country_collusion_all_three_estimate.csv", index=False
)
country_collusion_summary_df.head()

In [ ]:
# Estimated all-three circuit exposure to colluding countries.
fig, ax = plt.subplots(figsize=(10.5, 6.5))

for config in CONFIG_ORDER:
    sub = country_collusion_summary_df[
        country_collusion_summary_df["config"] == config
    ].sort_values("colluding_countries")
    lower = np.maximum(0, sub["all_three_circuits_pct"] - sub["all_three_circuits_std"].fillna(0))
    upper = np.minimum(100, sub["all_three_circuits_pct"] + sub["all_three_circuits_std"].fillna(0))
    ax.plot(
        sub["colluding_countries"], sub["all_three_circuits_pct"],
        color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
        linewidth=LINE_WIDTH, markersize=MARKER_SIZE,
        label=CONFIG_LABELS[config],
    )
    ax.fill_between(sub["colluding_countries"], lower, upper, color=CONFIG_COLORS[config], alpha=0.12)

ax.set_xlabel("Colluding countries", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Estimated all-three circuits (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_xticks(range(1, 11))
ax.set_ylim(0, 102)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=0))
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.legend(
    ncol=5, fontsize=LEGEND_SIZE, frameon=True,
    loc="lower center", bbox_to_anchor=(0.5, 1.02),
)
format_axes(ax)
fig.subplots_adjust(left=0.15, right=0.98, bottom=0.18, top=0.82)

out_path = FIGURES_DIR / "security_country_collusion_all_three_estimate.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()

In [ ]:
# Lower bound from colluding relay-hosting ASNs with same-AS Guard + Exit circuits.
guard_exit_provider_df = security_asn_match_df[
    security_asn_match_df["outcome"] == "Guard and exit"
].copy()

hosting_as_collusion_rows = []

for (network, config), matches in guard_exit_provider_df.groupby(["network", "config"]):
    asn_counts = matches.groupby("asn")["matches"].sum().sort_values(ascending=False)
    total_circuits = matches["total_circuits"].iloc[0]
    cumulative_matches = asn_counts.cumsum()
    for colluding_asns in range(1, min(10, len(cumulative_matches)) + 1):
        hosting_as_collusion_rows.append({
            "network": network,
            "config": config,
            "colluding_asns": colluding_asns,
            "same_host_as_circuit_pct": (
                cumulative_matches.iloc[colluding_asns - 1] / total_circuits * 100
            ),
        })

hosting_as_collusion_network_df = pd.DataFrame(hosting_as_collusion_rows)
hosting_as_collusion_summary_df = (
    hosting_as_collusion_network_df.groupby(["config", "colluding_asns"], as_index=False)
    .agg(
        same_host_as_circuit_pct=("same_host_as_circuit_pct", "mean"),
        same_host_as_circuit_std=("same_host_as_circuit_pct", "std"),
    )
)
hosting_as_collusion_summary_df.to_csv(
    FIGURES_DIR / "security_hosting_as_collusion_lower_bound.csv", index=False
)
hosting_as_collusion_summary_df.head()

In [ ]:
# Lower-bound circuit exposure to colluding relay-hosting ASNs.
fig, ax = plt.subplots(figsize=(10.5, 6.5))

for config in CONFIG_ORDER:
    sub = hosting_as_collusion_summary_df[
        hosting_as_collusion_summary_df["config"] == config
    ].sort_values("colluding_asns")
    lower = np.maximum(
        0, sub["same_host_as_circuit_pct"] - sub["same_host_as_circuit_std"].fillna(0)
    )
    upper = sub["same_host_as_circuit_pct"] + sub["same_host_as_circuit_std"].fillna(0)
    ax.plot(
        sub["colluding_asns"], sub["same_host_as_circuit_pct"],
        color=CONFIG_COLORS[config], marker=CONFIG_MARKERS[config],
        linewidth=LINE_WIDTH, markersize=MARKER_SIZE,
        label=CONFIG_LABELS[config],
    )
    ax.fill_between(
        sub["colluding_asns"], lower, upper,
        color=CONFIG_COLORS[config], alpha=0.12,
    )

ax.set_xlabel("Colluding relay-hosting ASNs", fontsize=LABEL_SIZE, labelpad=10)
ax.set_ylabel("Same-host-AS circuits (%)", fontsize=LABEL_SIZE, labelpad=10)
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100, decimals=1))
ax.tick_params(axis="both", labelsize=FONT_SIZE)
ax.grid(axis="both", linestyle="--", alpha=0.25)
ax.legend(ncol=3, fontsize=LEGEND_SIZE, frameon=True, loc="upper left")
format_axes(ax)
fig.subplots_adjust(left=0.16, right=0.98, bottom=0.18, top=0.97)

out_path = FIGURES_DIR / "security_hosting_as_collusion_lower_bound.png"
png_path, pdf_path = save_figure(fig, out_path, bbox_inches="tight", pad_inches=0.05)
print(png_path)
print(pdf_path)
plt.show()